### Example knowledge graph reasoning:

In [1]:
# imports
from llm_model_selection import local_LLM
from reasoning_mode_kg import langchain_reasoning
from node_ranking_kg import game_theory_node_ranking
from langchain_community.graphs import Neo4jGraph

# Setting up neo4j and LLM.
# Start database using
#       %NEO4J_HOME%\bin\neo4j console


import os
os.environ["NEO4J_URI"] = "bolt://localhost:7687"
os.environ["NEO4J_USERNAME"] = "anything"
os.environ["NEO4J_PASSWORD"] = "anything"
llm = local_LLM()

### Testing json to networkx

In [2]:
# Load graph into the class
networkx_example = langchain_reasoning(
    graph = "test_data/knowledge_graph.json",
    llm = llm,
    question = "which films has Brad Piit been in?",
    graph_type = "neo4j",
    display_names=["title", "name"]
    )

# Process json into networkx for langchain

networkx_example.langchain_json_networkx_processor()
networkx_example.select_example_prompts()

In [3]:
# Add nodes to nlp pipeline
networkx_example.nlp_pipeline()

c:\Users\natha\Documents\ACTICA\prototype\venv\Lib\site-packages\spacy\util.py:971: UserWarning: [W095] Model 'en_core_web_sm' (3.7.1) was trained with spaCy v3.7.2 and may not be 100% compatible with the current version (3.8.14). If you see errors or degraded performance, download a newer compatible model or retrain your custom model with the current spaCy version. For more details and available updates, run: python -m spacy validate
  warnings.warn(warn_msg)


In [4]:
# Quick test of the node ranking
# only works with json mode at the moment and should be moved to build_kg
ranking = game_theory_node_ranking(
    graph = networkx_example.graph,
    nodes = networkx_example.nodes,
    iterations = 1
)
sorted_rankings = sorted(ranking.items(), key=lambda item: item[1], reverse=True)
sorted_rankings[0]

iteration: 0 complete


('Bruce Willis', 16.0)

### Testing neo4j pipeline

In [5]:
graph = Neo4jGraph()
movies_query = """
LOAD CSV WITH HEADERS FROM 
'https://raw.githubusercontent.com/tomasonjo/blog-datasets/main/movies/movies_small.csv'
AS row
MERGE (m:Movie {id:row.movieId})
SET m.released = date(row.released),
    m.title = row.title,
    m.imdbRating = toFloat(row.imdbRating)
FOREACH (director in split(row.director, '|') | 
    MERGE (p:Person {name:trim(director)})
    MERGE (p)-[:DIRECTED]->(m))
FOREACH (actor in split(row.actors, '|') | 
    MERGE (p:Person {name:trim(actor)})
    MERGE (p)-[:ACTED_IN]->(m))
FOREACH (genre in split(row.genres, '|') | 
    MERGE (g:Genre {name:trim(genre)})
    MERGE (m)-[:IN_GENRE]->(g))
"""

graph.query(movies_query)
print(graph.schema)

C:\Users\natha\AppData\Local\Temp\ipykernel_16144\943105527.py:1: LangChainDeprecationWarning: The class `Neo4jGraph` was deprecated in LangChain 0.3.8 and will be removed in 1.0. An updated version of the class exists in the `langchain-neo4j package and should be used instead. To use it run `pip install -U `langchain-neo4j` and import as `from `langchain_neo4j import Neo4jGraph``.
  graph = Neo4jGraph()


Node properties:
Movie {imdbRating: FLOAT, id: STRING, released: DATE, title: STRING}
Person {name: STRING}
Genre {name: STRING}
Chunk {query: STRING, embedding: LIST, question: STRING, text: STRING, id: STRING}
Relationship properties:

The relationships:
(:Movie)-[:IN_GENRE]->(:Genre)
(:Person)-[:DIRECTED]->(:Movie)
(:Person)-[:ACTED_IN]->(:Movie)


In [6]:
# Initialise class objects.
neo4j_example = langchain_reasoning(
    graph = Neo4jGraph(),
    llm = llm,
    question = "what films was the actor Bradley Pierce in?",
    graph_type = "neo4j",
    display_names=["title", "name"]
    )

In [7]:
# Process for langchain
neo4j_example.langchain_neo4j_processor()
neo4j_example.select_example_prompts()

c:\Users\natha\Documents\ACTICA\prototype\reasoning_kg\processing_inputs_kg.py:68: UserWarning: 8 nodes are missing a valid display_name.this can cause graph information to be invisible to the query.
  warnings.warn(


In [8]:
# Add graph nodes to nlp_pipeline
neo4j_example.nlp_pipeline()
neo4j_example.question

'what films was the actor Bradley Pierce in?'

In [9]:
# Check for entities and return to user
adapted_question, successful_matches = neo4j_example.entity_checking_parser()
neo4j_example.question = adapted_question
print(adapted_question)

['Bradley Pierce', '?', 'what films', 'the actor']
what films was the actor Bradley Pierce in?


In [10]:
# prompt reasoning
neo4j_example.prompt_reasoning_parser()

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

--- Top examples for: 'what films was the actor Bradley Pierce in?' ---
1. Question: Which actors have worked in movies from both the comedy and action genres?
   Cypher:   MATCH (a:Person)-[:ACTED_IN]->(:Movie)-[:IN_GENRE]->(g1:Genre), (a)-[:ACTED_IN]->(:Movie)-[:IN_GENRE]->(g2:Genre) WHERE g1.name = 'Comedy' AND g2.name = 'Action' RETURN DISTINCT a.name

   Score:    0.9822396039962769

2. Question: Which actors played in the movie Casino?
   Cypher:   MATCH (m:Movie {{title: 'Casino'}})<-[:ACTED_IN]-(a) RETURN a.name

   Score:    0.9952160120010376

3. Question: Identify movies where directors also played a role in the film.
   Cypher:   MATCH (p:Person)-[:DIRECTED]->(m:Movie), (p)-[:ACTED_IN]->(m) RETURN m.title, p.name

   Score:    1.0156090259552002



> Entering new GraphCypherQAChain chain...
Generated Cypher:
MATCH (a:Person {name: 'Bradley Pierce'})-[:ACTED_IN]->(m:Movie) RETURN m.title

Full Context:
[{'m.title': 'Jumanji'}]

> Finished chain.


{'query': 'what films was the actor Bradley Pierce in?',
 'result': 'I do not know the answer.',
 'intermediate_steps': [{'query': "MATCH (a:Person {name: 'Bradley Pierce'})-[:ACTED_IN]->(m:Movie) RETURN m.title\n"},
  {'context': [{'m.title': 'Jumanji'}]}]}

In [11]:
response, df = neo4j_example.strict_based_parser()

Option: 2 selected for querying.
['Bradley Pierce', 'what films', 'the actor']
[3, 2]
3


The above is selecting the correct part of the data. But the data being parsed to the model is too great to summarize effectively. Definitely need an option to return the data. Or dynamic filtering which can just return the useful information. But I suspect this filtering will depend heavily on the specific data used.

In [13]:
df.head()

,search_node().element_id,search_node().labels,search_node().prop.name,start_node().element_id,start_node().labels,start_node().prop.name,relationship,end_node().element_id,end_node().labels,end_node().prop.imdbRating,end_node().prop.id,end_node().prop.title,end_node().prop.released,start_node().prop.imdbRating,start_node().prop.id,start_node().prop.title,start_node().prop.released,end_node().prop.name
0,4:8c815dd5-d3d9-4eaf-99aa-79cd05b6a384:599,frozenset({Person}),Bradley Pierce,4:8c815dd5-d3d9-4eaf-99aa-79cd05b6a384:599,frozenset({Person}),Bradley Pierce,ACTED_IN,4:8c815dd5-d3d9-4eaf-99aa-79cd05b6a384:2,frozenset({Movie}),6.9,2,Jumanji,1995-12-15,NaN,NaN,NaN,NaN,NaN
1,4:8c815dd5-d3d9-4eaf-99aa-79cd05b6a384:599,frozenset({Person}),Bradley Pierce,4:8c815dd5-d3d9-4eaf-99aa-79cd05b6a384:3,frozenset({Person}),Joe Johnston,DIRECTED,4:8c815dd5-d3d9-4eaf-99aa-79cd05b6a384:2,frozenset({Movie}),6.9,2,Jumanji,1995-12-15,NaN,NaN,NaN,NaN,NaN
2,4:8c815dd5-d3d9-4eaf-99aa-79cd05b6a384:599,frozenset({Person}),Bradley Pierce,4:8c815dd5-d3d9-4eaf-99aa-79cd05b6a384:598,frozenset({Person}),Robin Williams,ACTED_IN,4:8c815dd5-d3d9-4eaf-99aa-79cd05b6a384:2,frozenset({Movie}),6.9,2,Jumanji,1995-12-15,NaN,NaN,NaN,NaN,NaN
3,4:8c815dd5-d3d9-4eaf-99aa-79cd05b6a384:599,frozenset({Person}),Bradley Pierce,4:8c815dd5-d3d9-4eaf-99aa-79cd05b6a384:598,frozenset({Person}),Robin Williams,ACTED_IN,4:8c815dd5-d3d9-4eaf-99aa-79cd05b6a384:246,frozenset({Movie}),6.9,141,"Birdcage, The",1996-03-08,NaN,NaN,NaN,NaN,NaN
4,4:8c815dd5-d3d9-4eaf-99aa-79cd05b6a384:599,frozenset({Person}),Bradley Pierce,4:8c815dd5-d3d9-4eaf-99aa-79cd05b6a384:2,frozenset({Movie}),NaN,IN_GENRE,4:8c815dd5-d3d9-4eaf-99aa-79cd05b6a384:593,frozenset({Genre}),NaN,NaN,NaN,NaN,6.9,2,Jumanji,1995-12-15,Adventure
